# Análise dos resultados
Notebook para:
- Melhores parâmetros do Optuna por classificador
- Scatterplot de classificadores
- Boxplot de acurácia por classificador


In [2]:
import json, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter


In [ ]:
from pathlib import Path
import json
import pandas as pd

RESULTS_DIR = Path("../outputs") 

rows = []

for json_file in RESULTS_DIR.rglob("*.json"):
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        clf = data["classifier"]
        dataset = data["dataset"]
        run_id = data.get("run_id", json_file.stem)

        for user in data["user_results"]:
            rows.append({
                "dataset": dataset,
                "classifier": clf,
                "run_id": run_id,
                "user_id": user["user_id"],
                "accuracy": user["score"],
                "best_params": user.get("best_params", {})
            })

    except Exception as e:
        print(f"Erro ao ler {json_file}: {e}")

df = pd.DataFrame(rows)

print(f"Total de registros: {len(df)}")
print(f"Classificadores: {df['classifier'].nunique()}")
display(df.head())

Total de registros: 0


KeyError: 'classifier'

## 1. Melhores parâmetros do Optuna por classificador

In [ ]:
def most_common_params(series):
    counters = {}
    for params in series:
        for k, v in params.items():
            counters.setdefault(k, Counter())[str(v)] += 1
    out = {}
    for k, c in counters.items():
        out[k] = c.most_common(1)[0][0]
    return out

best_params_by_classifier = (
    df.groupby('classifier')['best_params']
      .apply(most_common_params)
)

for clf, params in best_params_by_classifier.items():
    print('\n' + '='*80)
    print(clf)
    print('='*80)
    for k, v in params.items():
        print(f'{k}: {v}')


## 2. Scatterplot: classificador X vs classificador Y

In [ ]:
classifier_x = 'random-forest'
classifier_y = 'xgboost'  # ajuste

pivot = df.pivot_table(
    index=['dataset','user_id'],
    columns='classifier',
    values='accuracy',
    aggfunc='mean'
)

tmp = pivot[[classifier_x, classifier_y]].dropna()

plt.figure(figsize=(6,6))
plt.scatter(tmp[classifier_x], tmp[classifier_y])
plt.xlabel(classifier_x)
plt.ylabel(classifier_y)
plt.title(f'{classifier_x} vs {classifier_y}')
plt.grid(True)
plt.show()


## 3. Boxplot de acurácia por classificador

In [ ]:
order = (
    df.groupby('classifier')['accuracy']
      .median()
      .sort_values(ascending=False)
      .index
)

data = [df.loc[df['classifier']==c, 'accuracy'] for c in order]

plt.figure(figsize=(10,5))
plt.boxplot(data, tick_labels=order)
plt.ylabel('Accuracy')
plt.xlabel('Classifier')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
